In [7]:
import os
import glob
import numpy as np
import pandas as pd

import joblib
from sklearn.preprocessing import StandardScaler

In [4]:
DATA_DIR = "./airkorea_result"


In [ ]:
def preprocess_pm(raw_df, pm_type="pm10"):
    df = raw_df.copy()
    hour_cols = [col for col in df.columns if col.endswith("시")]

    candidate_id_vars = ["date", "측정망", "측정소명", "지점ID", "지점"]
    id_vars = [c for c in candidate_id_vars if c in df.columns]

    df_long = df.melt(
        id_vars=id_vars,
        value_vars=hour_cols,
        var_name="hour",
        value_name=pm_type
    )

    df_long["hour"] = (
        df_long["hour"]
        .astype(str)
        .str.replace("시", "", regex=False)
        .astype(int)
    )

    df_long[pm_type] = (
        df_long[pm_type]
        .replace("-", np.nan)
        .astype(float)
    )

    df_long["date"] = pd.to_datetime(df_long["date"])

    df_long["station"] = np.nan
    for col in ["지점ID", "측정소명", "지점", "측정망"]:
        if col in df_long.columns:
            df_long["station"] = df_long["station"].fillna(df_long[col])

    df_long = df_long.dropna(subset=["station"])

    df_long = df_long.sort_values(["station", "date", "hour"])

    df_long[pm_type] = df_long.groupby("station")[pm_type].transform(
        lambda s: s.interpolate(limit_direction="both")
    )
    df_long = df_long.dropna(subset=[pm_type])

    df_long["month"] = df_long["date"].dt.month
    df_long["day"] = df_long["date"].dt.day
    df_long["weekday"] = df_long["date"].dt.weekday  # 월=0, 일=6
    df_long["is_weekend"] = (df_long["weekday"] >= 5).astype(int)

    # 측정소별 lag/rolling feature
    g = df_long.groupby("station")

    df_long["lag1"] = g[pm_type].shift(1)
    df_long["lag24"] = g[pm_type].shift(24)
    df_long["rolling3"] = g[pm_type].transform(
        lambda s: s.rolling(3, min_periods=1).mean()
    )
    df_long["rolling24"] = g[pm_type].transform(
        lambda s: s.rolling(24, min_periods=1).mean()
    )

    # lag1이 있는 행만 사용 ---
    df_model = df_long.dropna(subset=["lag1"]).copy()

    for col in ["lag24", "rolling3", "rolling24"]:
        if col in df_model.columns:
            mean_val = df_model[col].mean()
            df_model[col] = df_model[col].fillna(mean_val)

    num_feature_cols = [
        # "hour",  
        "month",
        "day",
        "weekday",
        "is_weekend",
        "lag1",
        "lag24",
        "rolling3",
        "rolling24",
    ]
    num_feature_cols = [c for c in num_feature_cols if c in df_model.columns]

    if len(df_model) == 0:
        raise ValueError(f"전처리 결과 행이 0개입니다. station/{pm_type}/lag1을 다시 확인하세요.")

    scaler = StandardScaler()
    df_model[num_feature_cols] = scaler.fit_transform(df_model[num_feature_cols])

    # One-Hot Encoding 추가
    # station_dummies = pd.get_dummies(df_model["station"], prefix="station", drop_first=True)
    # df_model = pd.concat([df_model, station_dummies], axis=1)
    
    return df_model, scaler#, station_dummies

In [9]:
for pm_type in ["pm2.5"]:
    all_files = glob.glob(os.path.join(DATA_DIR, pm_type, "*.csv"))

    dfs = []
    for f in all_files:
        # 인코딩
        try:
            df = pd.read_csv(f, encoding="cp949")
        except UnicodeDecodeError:
            df = pd.read_csv(f, encoding="utf-8")
        dfs.append(df)

    raw = pd.concat(dfs, ignore_index=True)
    print("원본 데이터 크기:", raw.shape)
    df_model, scaler = preprocess_pm(raw, pm_type=pm_type)
    print("전처리 후 데이터 크기:", df_model.shape)

    # (날짜 기준으로 시계열 split – 필요에 맞게 조건 수정)
    split_date = "2025-07-01"
    train = df_model[df_model["date"] < split_date]
    test  = df_model[df_model["date"] >= split_date]

    feature_cols = [
        "hour", "month", "day", "weekday", "is_weekend",
        "lag1", "lag24", "rolling3", "rolling24",
    ]

    X_train = train[feature_cols]
    y_train = train[pm_type]

    X_test = test[feature_cols]
    y_test = test[pm_type]
    train.to_csv(f"train_{pm_type}.csv", index=False)
    test.to_csv(f"test_{pm_type}.csv", index=False)
    df_model.to_csv(f"preprocessed_{pm_type}.csv", index=False)

    joblib.dump(scaler, f"scaler_{pm_type}.pkl")

원본 데이터 크기: (469869, 29)
전처리 후 데이터 크기: (11225141, 16)
